In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

class SparkConfig :
    def creat_sparksession() :
        spark = SparkSession.builder.config("spark.driver.memory" , "8g") \
                            .appName("clean data") \
                            .master("local[*]") \
                            .getOrCreate()
        spark.sparkContext._jsc.hadoopConfiguration().set("fs.defaultFS", "file:///")
        return spark

In [2]:
spark = SparkConfig.creat_sparksession()

In [3]:
class DataLoader :
    def __init__(selt) :
        selt.spark = SparkConfig.create_sparksession() 
    def clean_works() :    
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/works/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        list_file = []
        list_col = []
        for file in files :
            works = spark.read.format("parquet").load(file) 
            
            works = works.withColumn("cover_id" , col("cover_id").cast("long"))
            for c in works.columns :
                if c not in list_col :
                    list_col.append(c)
            for c in list_col :
                if c not in works.columns :
                    works = works.withColumn(c , lit(None).cast("string"))
            list_file.append(works)
        works = reduce(lambda a,b : a.union(b) , list_file)
        return works

    def clean_editions() :    
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/editions/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        list_file = []
        list_col = []
        for file in files :
            editions = spark.read.format("parquet").load(file) 
            if "number_of_pages" in editions.columns :
                editions = editions.withColumn("number_of_pages" , col("number_of_pages").cast("long"))
            
            for c in editions.columns :
                if c not in list_col :
                    list_col.append(c)
            for c in list_col :
                if c not in editions.columns :
                    editions = editions.withColumn(c , lit(None).cast("string"))
            list_file.append(editions)
        editions = reduce(lambda a,b : a.union(b) , list_file)
        editions.limit(5).coalesce(1).write.mode("overwrite").parquet("s3://mhai-bk/data/editions/")
        return editions 
    
    def clean_works() :        
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/authors/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        for file in files :
            authors = spark.read.format("parquet").load(file) 
            if "number_of_pages" in authors.columns :
                authors = authors.withColumn("number_of_pages" , col("number_of_pages").cast("long"))
            
            list_file.append(authors)
        authors = reduce(lambda a,b : a.unionByName(b , allowMissingColumns = True) , list_file)
        authors.limit(5).coalesce(1).write.mode("overwrite").parquet("s3://mhai-bk/data/authors/")
        return authors

In [4]:
spark

In [5]:
works = spark.read.parquet("data/works.parquet")

In [6]:
editions = spark.read.parquet("data/editions.parquet")

In [7]:
authors = spark.read.parquet("data/authors.parquet")

In [8]:
# Chọn các cột cần để phân tích
works = works.select("key", "title", "first_publish_year", "edition_count", "subject", "authors" )

In [9]:
works.printSchema()

root
 |-- key: string (nullable = true)
 |-- title: string (nullable = true)
 |-- first_publish_year: double (nullable = true)
 |-- edition_count: long (nullable = true)
 |-- subject: string (nullable = true)
 |-- authors: string (nullable = true)



In [10]:
works.select([count(when (col(c).isNull() , c)).alias(c) for c in works.columns ]).show()

+---+-----+------------------+-------------+-------+-------+
|key|title|first_publish_year|edition_count|subject|authors|
+---+-----+------------------+-------------+-------+-------+
|  0|    0|               145|            0|      0|      0|
+---+-----+------------------+-------------+-------+-------+



In [11]:
works.show()

+-----------------+--------------------+------------------+-------------+--------------------+--------------------+
|              key|               title|first_publish_year|edition_count|             subject|             authors|
+-----------------+--------------------+------------------+-------------+--------------------+--------------------+
|  /works/OL21177W|   Wuthering Heights|            1846.0|         2886|["form:novel", "g...|[{"key": "/author...|
|  /works/OL66513W|                Emma|            1815.0|         2263|["Social life and...|[{"key": "/author...|
|  /works/OL66562W|Sense and Sensibi...|            1811.0|         2090|["Fiction, Romanc...|[{"key": "/author...|
|  /works/OL29983W|        Little Women|            1848.0|         1888|["Romans", "Jeune...|[{"key": "/author...|
| /works/OL267096W|       Анна Каренина|            1876.0|         1314|["Fiction", "Adul...|[{"key": "/author...|
|/works/OL1095427W|           Jane Eyre|            1847.0|         1170

In [12]:
# Làm sạch cột key và first_pubnlish_year 
work_key = works.withColumn("key" , regexp_extract(col("key") , r"[A-Z]+\d+[A-Z]", 0)) \
                .withColumnRenamed("key", "work_id") \
                .withColumn("first_publish_year" , col("first_publish_year").cast("int")) # chuyển đổi kiểu dữ liệu cột first_publish_year

In [13]:
# Xử lý giá trị missing value bằng cách lấy trung vị
median = work_key.approxQuantile("first_publish_year", [0.5], 0.01)[0]
work_key = work_key.fillna({"first_publish_year" : int(median)})

In [14]:
# Tạo ra bảng dim_work 
dim_work = work_key.select("work_id", "title", "first_publish_year", "edition_count").dropDuplicates()

In [15]:
# Explode cột subject 
work_explode_sub = work_key.withColumn("subject" , regexp_replace("subject" , r"[\]\[]", "")) \
                           .withColumn("subject", explode(split("subject" , ",")))

In [16]:
# Làm sạch cột subject
work_explode_sub = work_explode_sub.withColumn("subject", regexp_replace("subject", r'form:|genre:|"', ""))

In [17]:
# Làm sạch bảng work_explode_sub
work_explode_sub = work_explode_sub.filter(col("subject").isNotNull() & ~col("subject").rlike("\\\\|[0-9]"))
# Xóa bỏ khoảng trắng, lọc các dữ liệu sạch
work_explode_sub = work_explode_sub.orderBy("subject") \
           .withColumn("subject", trim(col("subject"))) \
           .filter(~col("subject").rlike("&|.com$|@|^:|^\\.") & trim(col("subject") != "") ) \
           .withColumn("subject", regexp_replace("subject" ,"'|\\*|-" , "")) 

In [18]:
# Tạo bảng dim_subject
dim_subject = work_explode_sub.select("subject").dropDuplicates()

In [19]:
dim_subject = dim_subject.distinct()

In [20]:
dim_subject.count()

43986

In [21]:
# Gen Id cho bảng dim_subject sử dụng window function
dim_subject = dim_subject.withColumn("subject_id", row_number().over(Window.orderBy("subject")))

In [22]:
dim_subject.show(5)

+--------------------+----------+
|             subject|subject_id|
+--------------------+----------+
|             Earl of|         1|
|             Fiction|         2|
|     King of England|         3|
| North Carolina  ...|         4|
|    Queen of England|         5|
+--------------------+----------+
only showing top 5 rows


In [23]:
# Tạo bảng bridge work_subject
work_subject = work_explode_sub.join(dim_subject , on = "subject", how = "inner") \
                .select("work_id", "subject_id").orderBy("subject")

In [24]:
work_subject.show(10)

+-----------+----------+
|    work_id|subject_id|
+-----------+----------+
| OL1652426W|         1|
|   OL19870W|         2|
| OL5361840W|         3|
|   OL54797W|         4|
| OL1652426W|         5|
|  OL261159W|         6|
| OL1652379W|         7|
|OL13648986W|         8|
| OL2582567W|         9|
| OL8887370W|         9|
+-----------+----------+
only showing top 10 rows


In [25]:
work_explode_sub.select("authors").show(5, truncate = False)

+---------------------------------------------------------------------------------------------------------------------------+
|authors                                                                                                                    |
+---------------------------------------------------------------------------------------------------------------------------+
|[{"key": "/authors/OL390519A", "name": "Alice McGill"}]                                                                    |
|[{"key": "/authors/OL225331A", "name": "Colleen McCullough"}, {"key": "/authors/OL9465374A", "name": "Colleen McCullough"}]|
|[{"key": "/authors/OL163442A", "name": "Sir Sidney Lee"}]                                                                  |
|[{"key": "/authors/OL21594A", "name": "Jane Austen"}]                                                                      |
|[{"key": "/authors/OL365212A", "name": "Murasaki Shikibu"}]                                                          

In [26]:
# Tạo author schema
author_schema = ArrayType(
    StructType([
        StructField("key", StringType(), True),
        StructField("name", StringType(), True)
    ])
)

In [27]:
# Trích ra cột id và name của author
work_author = work_explode_sub.withColumn("authors" , from_json(col("authors"), author_schema)) \
                .withColumn("authors", explode(col("authors"))) \
                .select("work_id","authors.key")

In [28]:
work_author.show(5)

+----------+-------------------+
|   work_id|                key|
+----------+-------------------+
|OL2676055W| /authors/OL390519A|
|OL1882556W| /authors/OL225331A|
|OL1882556W|/authors/OL9465374A|
|OL1540137W| /authors/OL163442A|
|  OL66562W|  /authors/OL21594A|
+----------+-------------------+
only showing top 5 rows


In [29]:
# Làm sạch cột author_id
work_author = work_author.withColumn("key", regexp_extract(col("key"), r"[A-Z]+\d+[A-Z]", 0))

In [30]:
work_author = work_author.select(col("work_id"), col("key").alias("author_id"))

In [31]:
work_author.show(5)

+----------+----------+
|   work_id| author_id|
+----------+----------+
|OL2676055W| OL390519A|
|OL1882556W| OL225331A|
|OL1882556W|OL9465374A|
|OL1540137W| OL163442A|
|  OL66562W|  OL21594A|
+----------+----------+
only showing top 5 rows


## Edition

In [32]:
editions = editions.select("key", "works", "title", "publish_date", "publishers", "languages", "number_of_pages", "isbn_13")

In [33]:
editions.show(5)

+------------------+--------------------+--------------------+------------+--------------------+--------------------+---------------+-----------------+
|               key|               works|               title|publish_date|          publishers|           languages|number_of_pages|          isbn_13|
+------------------+--------------------+--------------------+------------+--------------------+--------------------+---------------+-----------------+
|/books/OL61212055M|[{"key": "/works/...|O Morro dos Vento...|        2014|   ["Martin Claret"]|[{"key": "/langua...|            524|["9788544000182"]|
|/books/OL56513356M|[{"key": "/works/...| Hauts de Hurle-Vent|        2017|["CreateSpace Ind...|[{"key": "/langua...|            356|["9781974501502"]|
|/books/OL56523458M|[{"key": "/works/...|  Hauts de Hurlevent|        2017|["CreateSpace Ind...|[{"key": "/langua...|            368|["9781981584758"]|
|/books/OL56296338M|[{"key": "/works/...| Hauts de Hurle-Vent|        2017|["CreateSpace

In [34]:
editions.select("works").show(5, truncate = False)

+----------------------------+
|works                       |
+----------------------------+
|[{"key": "/works/OL21177W"}]|
|[{"key": "/works/OL21177W"}]|
|[{"key": "/works/OL21177W"}]|
|[{"key": "/works/OL21177W"}]|
|[{"key": "/works/OL21177W"}]|
+----------------------------+
only showing top 5 rows


In [35]:
# Tạo work schema
work_schema = ArrayType(
    StructType([
        StructField("key", StringType(), True)
    ])
)

In [36]:
# Làm sạch cột works trích xuất ra work_id
editions_clean_work = editions.withColumn("works", from_json("works", work_schema)) \
                              .withColumn("works", regexp_extract(col("works")[0]["key"], r"[A-Z]+\d+[A-Z]", 0))

In [37]:
editions_clean_work.show(5)

+------------------+--------+--------------------+------------+--------------------+--------------------+---------------+-----------------+
|               key|   works|               title|publish_date|          publishers|           languages|number_of_pages|          isbn_13|
+------------------+--------+--------------------+------------+--------------------+--------------------+---------------+-----------------+
|/books/OL61212055M|OL21177W|O Morro dos Vento...|        2014|   ["Martin Claret"]|[{"key": "/langua...|            524|["9788544000182"]|
|/books/OL56513356M|OL21177W| Hauts de Hurle-Vent|        2017|["CreateSpace Ind...|[{"key": "/langua...|            356|["9781974501502"]|
|/books/OL56523458M|OL21177W|  Hauts de Hurlevent|        2017|["CreateSpace Ind...|[{"key": "/langua...|            368|["9781981584758"]|
|/books/OL56296338M|OL21177W| Hauts de Hurle-Vent|        2017|["CreateSpace Ind...|[{"key": "/langua...|            356|["9781548006495"]|
|/books/OL55977620M|

In [38]:
# Chuyển đổi kiểu dữ liệu publish_date và làm sạch cột key (cột edition_id)
editions_clean = editions_clean_work.withColumn("publish_date", col("publish_date").cast("string")) \
                   .withColumn("key", regexp_extract("key", r"[A-Z]+\d+[A-Z]", 0))

In [39]:
editions_clean.printSchema()

root
 |-- key: string (nullable = true)
 |-- works: string (nullable = true)
 |-- title: string (nullable = true)
 |-- publish_date: string (nullable = true)
 |-- publishers: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- number_of_pages: long (nullable = true)
 |-- isbn_13: string (nullable = true)



In [40]:
editions_clean.show(5)

+-----------+--------+--------------------+------------+--------------------+--------------------+---------------+-----------------+
|        key|   works|               title|publish_date|          publishers|           languages|number_of_pages|          isbn_13|
+-----------+--------+--------------------+------------+--------------------+--------------------+---------------+-----------------+
|OL61212055M|OL21177W|O Morro dos Vento...|        2014|   ["Martin Claret"]|[{"key": "/langua...|            524|["9788544000182"]|
|OL56513356M|OL21177W| Hauts de Hurle-Vent|        2017|["CreateSpace Ind...|[{"key": "/langua...|            356|["9781974501502"]|
|OL56523458M|OL21177W|  Hauts de Hurlevent|        2017|["CreateSpace Ind...|[{"key": "/langua...|            368|["9781981584758"]|
|OL56296338M|OL21177W| Hauts de Hurle-Vent|        2017|["CreateSpace Ind...|[{"key": "/langua...|            356|["9781548006495"]|
|OL55977620M|OL21177W|        Emily Brontë|        2016|["CreateSpace

In [41]:
editions_clean.select("publish_date").orderBy("publish_date").show(30)

+------------+
|publish_date|
+------------+
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
+------------+
only showing top 30 rows


In [42]:
# Làm sạch cột publish_date
publish_date_clean = editions_clean.withColumn("publish_date", trim(regexp_extract("publish_date", r"(\d{4})", 1))) \
                                    .withColumn("publish_date", when(col("publish_date") == "", None)
                                                                .otherwise(col("publish_date").cast("int"))) 

In [43]:
publish_date_clean.select("publish_date").orderBy("publish_date").show(10000)

+------------+
|publish_date|
+------------+
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        N

In [44]:
# Lấy trung vị cột publish_date
publish_date_median = publish_date_clean.approxQuantile("publish_date", [0.5], 0.01)[0]


In [45]:
# Lấy trung vị cột number_of_pages
number_of_pages_median = publish_date_clean.approxQuantile("number_of_pages", [0.5], 0.01)[0]


In [46]:
# Làm sạch giá trị null các cột
editions_clean = publish_date_clean.fillna({"publish_date" : publish_date_median, "number_of_pages" : number_of_pages_median,
                                            "publishers" : "unknown", "languages" : "unknown", "title" : "unknown"})

In [47]:
editions_clean.show(5)

+-----------+--------+--------------------+------------+--------------------+--------------------+---------------+-----------------+
|        key|   works|               title|publish_date|          publishers|           languages|number_of_pages|          isbn_13|
+-----------+--------+--------------------+------------+--------------------+--------------------+---------------+-----------------+
|OL61212055M|OL21177W|O Morro dos Vento...|        2014|   ["Martin Claret"]|[{"key": "/langua...|            524|["9788544000182"]|
|OL56513356M|OL21177W| Hauts de Hurle-Vent|        2017|["CreateSpace Ind...|[{"key": "/langua...|            356|["9781974501502"]|
|OL56523458M|OL21177W|  Hauts de Hurlevent|        2017|["CreateSpace Ind...|[{"key": "/langua...|            368|["9781981584758"]|
|OL56296338M|OL21177W| Hauts de Hurle-Vent|        2017|["CreateSpace Ind...|[{"key": "/langua...|            356|["9781548006495"]|
|OL55977620M|OL21177W|        Emily Brontë|        2016|["CreateSpace

In [48]:
editions_clean.select([count(when (col(c).isNull() , c)).alias(c) for c in editions.columns ]).show()

+---+-----+-----+------------+----------+---------+---------------+-------+
|key|works|title|publish_date|publishers|languages|number_of_pages|isbn_13|
+---+-----+-----+------------+----------+---------+---------------+-------+
|  0|    0|    0|           0|         0|        0|              0|  59642|
+---+-----+-----+------------+----------+---------+---------------+-------+



In [49]:
editions_clean.count()

224540

In [53]:
# Làm sạch cột title
editions_clean = editions_clean.filter(col("title").rlike("[A-Za-z]")) \
                                .withColumnRenamed("key", "edition_id") \
                                .withColumnRenamed("works", "work_id")

In [54]:
editions_clean.show(5)

+-----------+--------+--------------------+------------+--------------------+--------------------+---------------+-----------------+
| edition_id| work_id|               title|publish_date|          publishers|           languages|number_of_pages|          isbn_13|
+-----------+--------+--------------------+------------+--------------------+--------------------+---------------+-----------------+
|OL61212055M|OL21177W|O Morro dos Vento...|        2014|   ["Martin Claret"]|[{"key": "/langua...|            524|["9788544000182"]|
|OL56513356M|OL21177W| Hauts de Hurle-Vent|        2017|["CreateSpace Ind...|[{"key": "/langua...|            356|["9781974501502"]|
|OL56523458M|OL21177W|  Hauts de Hurlevent|        2017|["CreateSpace Ind...|[{"key": "/langua...|            368|["9781981584758"]|
|OL56296338M|OL21177W| Hauts de Hurle-Vent|        2017|["CreateSpace Ind...|[{"key": "/langua...|            356|["9781548006495"]|
|OL55977620M|OL21177W|        Emily Brontë|        2016|["CreateSpace

In [55]:
# Tạo bảng dim_editions
dim_edition = editions_clean.select("edition_id", "title", "publish_date", "publishers", "languages").dropDuplicates() 

In [56]:
dim_edition.show(5)

+-----------+--------------------+------------+--------------------+--------------------+
| edition_id|               title|publish_date|          publishers|           languages|
+-----------+--------------------+------------+--------------------+--------------------+
|OL61212055M|O Morro dos Vento...|        2014|   ["Martin Claret"]|[{"key": "/langua...|
|OL38138835M|Room with a View ...|        2021|["Independently P...|[{"key": "/langua...|
|OL55521569M|    Room with a View|        2016|["CreateSpace Ind...|[{"key": "/langua...|
|OL55738986M|    Room with a View|        2016|["CreateSpace Ind...|[{"key": "/langua...|
|OL54816473M|    Room with a View|        2014|["CreateSpace Ind...|[{"key": "/langua...|
+-----------+--------------------+------------+--------------------+--------------------+
only showing top 5 rows


# Author


In [ ]:
authors.select([count(when (col(c).isNull() , c)).alias(c) for c in authors.columns ]).show()

+----+------+----------+-------------+----+----------+---------------+----------+------+--------------+---+------+---------------+--------+-------+-------------+-----+------+--------+-------+------+-----------+-----------+
|name|   bio|remote_ids|personal_name|type|birth_date|alternate_names|death_date| title|source_records|key|photos|latest_revision|revision|created|last_modified|   id| links|location|comment|  role|fuller_name|entity_type|
+----+------+----------+-------------+----+----------+---------------+----------+------+--------------+---+------+---------------+--------+-------+-------------+-----+------+--------+-------+------+-----------+-----------+
|1351|118879|    116277|        98199|   0|    114660|         114206|    116978|122920|         59978|  0|119942|          29544|       0|  26975|            0|95543|123295|  124440| 125273|125279|     125480|     125494|
+----+------+----------+-------------+----+----------+---------------+----------+------+--------------+---+-

In [ ]:
authors.count()

125500